# Лабораторная работа 2. Исследовательский анализ данных

**Вариант 14.** Набор данных `clients.csv` содержит сведения о клиентах магазина: идентификатор, год рождения, образование, семейное положение, доход семьи, количество детей, дату регистрации и количество покупок по акциям.

Цель работы — исследовать распределения и взаимосвязи признаков средствами pandas, matplotlib и seaborn.


## 1 Загрузка и предварительная обработка


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('clients.csv', sep=';')
df.info()
df.describe(include='all')


In [ ]:
df.columns = df.columns.str.lower()
df = df.drop_duplicates().reset_index(drop=True)
df['education'] = df['education'].str.strip().replace({'2n Cycle': 'Graduation'})
df['marital_status'] = df['marital_status'].str.strip().replace({'MARRIED':'Married', 'SINGL':'Single', 'Alone':'Single'})
df = df.dropna(subset=['dt_customer']).reset_index(drop=True)
for col in ['income', 'kidhome', 'numdealspurchases']:
    df[col] = df[col].fillna(df[col].median())
df['dt_customer'] = pd.to_datetime(df['dt_customer'], format='%d.%m.%Y')
df['income'] = df['income'].round().astype(int)
df['kidhome'] = df['kidhome'].round().astype(int)
df['numdealspurchases'] = df['numdealspurchases'].round().astype(int)
df['income_level'] = pd.qcut(df['income'], q=3, labels=['Низкий', 'Средний', 'Высокий'])
df.head()


## 2 Матрица рассеяния и категориальная диаграмма


In [ ]:
pd.plotting.scatter_matrix(df[['year_birth','income','kidhome','numdealspurchases']], figsize=(10,10), diagonal='hist')
plt.show()
sns.scatterplot(data=df, x='year_birth', y='income', hue='education')
plt.show()


Матрица показывает попарные связи числовых признаков. Категориальная диаграмма позволяет сравнить доходы клиентов разных образовательных групп.


## 3 Гистограммы, корреляция и ковариация


In [ ]:
numeric = ['year_birth','income','kidhome','numdealspurchases']
df[numeric].hist(bins=12, figsize=(12,8)); plt.show()
print('Матрица корреляции:')
print(df[numeric].corr().round(3))
print('Матрица ковариации:')
print(df[numeric].cov().round(2))
sns.heatmap(df[numeric].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.show()


## 4 Дополнительные виды графиков


In [ ]:
df.plot(x='income', y='numdealspurchases', kind='hexbin', gridsize=18, cmap='viridis', figsize=(9,6), sharex=False)
plt.show()
sns.boxplot(x=df['income']); plt.show()
sns.boxplot(data=df, x='income_level', y='income', hue='income_level', legend=False); plt.show()
sns.boxplot(data=df, x='education', y='income', hue='education', legend=False); plt.show()
df.boxplot(column='income', by='marital_status', figsize=(10,6)); plt.suptitle(''); plt.show()


## 5 Задания варианта 14


In [ ]:
# Задание 1: количество клиентов по образованию и семейному статусу
task1 = df.groupby(['education', 'marital_status']).size().reset_index(name='count')
print(task1)
sns.countplot(data=df, x='education', hue='marital_status'); plt.show()

# Задание 2: средний доход по семейному положению
task2 = df.pivot_table(index='marital_status', values='income', aggfunc='mean').round(2)
print(task2)
task2.plot(kind='line', linestyle='', marker='*', markersize=20, color='deeppink', figsize=(10,6)); plt.show()

# Задание 3: образование клиентов с детьми
task3 = df[df['kidhome'] > 0]['education'].value_counts()
print(task3)
task3.plot(kind='pie', autopct='%1.1f%%', startangle=90, figsize=(10,6)); plt.ylabel(''); plt.show()


## 6 Вывод


После загрузки и очистки данных были обработаны пропуски, дубликаты, категориальные значения и дата регистрации. Распределения доходов и количества покупок изучены с помощью гистограмм и boxplot, а взаимосвязи — с помощью scatter plot, hexbin и матрицы корреляций. Для варианта 14 построены требуемые группировка, сводный график и круговая диаграмма.
